# Session 5 — Exercises

**Before session 6.**

Same house pattern: **predict, then measure.** Write down what you expect before you run anything —
three of tonight's four exercises have an outcome that surprised the person who wrote them, and you
only get to be surprised if you committed to a prediction first.

Answers go in the **✍️ Your answer** cells. Code TODOs are marked `# TODO`.

Solutions are released with session 6, after these have been reviewed in class.

## Setup — given, just run it

In [ ]:
import json
import sys
from pathlib import Path

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                      # noqa: E402  — also loads .env
from shopwise_data import ORDERS, PRODUCTS, SHIPMENTS   # noqa: E402
from tools import TOOLS                          # noqa: E402  — the SHIPPED tools, not a copy
import agent as project_agent                    # noqa: E402  — the shipped agent

model = init_chat_model(settings.model_id, temperature=0)

print(f"{len(TOOLS)} tools · {settings.model_id}")
print([t.name for t in TOOLS])


def tools_called(result) -> list:
    """Every tool request in a finished run, in order. Used by several exercises."""
    return [(tc["name"], tc["args"])
            for m in result["messages"]
            for tc in (getattr(m, "tool_calls", None) or [])]

## ⭐ Exercise 1 — Docstring surgery

**What this is really teaching:** that the four parts of a tool are *redundant on purpose*, and that
when routing degrades it degrades **silently** — the agent still answers, still sounds confident, and
is still grounded in a real tool result. It is simply a worse answer, and nothing in your program
raises anything.

The question, throughout: *"I'm tanvir@example.com. Where is my parcel for ORD-5003 and when will it
arrive?"* The good answer names SwiftEx, tracking SW-88231, and an ETA of 2026-08-08 — all of which
live behind `get_shipment`.

You will run it three times, degrading the tool one part at a time:

1. **as shipped** — the baseline;
2. **docstring gutted** — `get_shipment` keeps its name and signature, but its docstring becomes
   `"Gets data."`;
3. **name gutted too** — the same useless docstring on a tool called `helper(x: str)`.

Predict all three before you run any of them.

**✍️ Your prediction:**

*(For each of the three: does the agent call the shipment tool? Does the customer get the tracking
number and the ETA? Write it down now.)*

In [ ]:
Q = "I'm tanvir@example.com. Where is my parcel for ORD-5003 and when will it arrive?"

GROUNDED = ("You are ShopWise's support assistant. Answer ONLY from tool results. "
            "If you have not looked something up, you do not know it.")


def shipment_body(order_id: str) -> dict:
    """The identical body for all three variants — only the DESCRIPTION changes."""
    s = SHIPMENTS.get(order_id.strip().upper())
    if s is None:
        return {"error": f"no shipment yet for order {order_id}"}
    return {"order_id": order_id.strip().upper(), **s}


# TODO 1: build the two degraded tools with @tool.
#         - `get_shipment(order_id: str) -> dict` with the docstring "Gets data."
#         - `helper(x: str) -> dict` with the same docstring
#         Both must call shipment_body, so the ONLY difference is what the model reads.

# TODO 2: build three agents — TOOLS as shipped, then TOOLS with get_shipment swapped for each
#         degraded version. `[t for t in TOOLS if t.name != "get_shipment"]` gives you the rest.

# TODO 3: run Q through all three. For each, print tools_called(result) and the final answer.

**✍️ Your answer.**

- Which of the three levels actually broke it? Was that where you predicted?
- In the level that broke, **what did the agent do instead** — and would you have noticed the answer
  was worse if you had not known what the good answer looked like?
- The step that did *not* break is the interesting one. Name the part of the tool that carried the
  routing on its own, and say what that tells you about which part to fix first when an agent ignores
  a tool.
- Now the design question. If the four parts back each other up, why write a careful docstring at
  all? Answer it in terms of a tool whose name **cannot** carry the meaning — invent one from
  ShopWise's domain where two tools would sound alike.
- One sentence for your PR checklist: what would you ask a colleague to change about a tool named
  `helper` in review?

## ⭐⭐ Exercise 2 — A new tool, from scratch

**What this is really teaching:** the four-part anatomy as something you *do*, not something you
recognise — and that the success case for a lookup tool is often an honest **no**.

Write `check_stock(sku_or_name: str) -> dict` and give it to an agent, then ask *"is the kettle
available?"*

The kettle is `SKU-300`, and the fixture gives it `stock: 0` on purpose. So a correct run ends with
the agent telling the customer the kettle is out of stock — not with an apology about being unable to
check, and not with a cheerful invention of availability.

Work against the checklist rather than from memory:

| part | your job |
|---|---|
| name | a verb, and one that could not be confused with `get_product` |
| arguments | typed. One string, taking a SKU **or** part of a name, like `get_product` |
| docstring | when to use it, what comes back, and what `stock: 0` means |
| return | plain data. Include enough for the agent to answer without a second call |

**✍️ Your prediction:**

*(Will the agent call your tool, or `get_product`? They overlap. What decides?)*

In [ ]:
# TODO 1: write check_stock with @tool. PRODUCTS is imported for you above; look at how
#         project/tools.py does the SKU-or-name match rather than inventing a new rule.

# TODO 2: build an agent with TOOLS + [check_stock] and the GROUNDED system prompt.

# TODO 3: ask "is the kettle available?" — print tools_called(result) and the final answer.

# TODO 4: ask "is the kettle available, and if not what else do you sell?" and look at whether
#         the agent chains a second call. Nothing told it to.

**✍️ Your answer.**

- Did the agent choose `check_stock` or `get_product`? Quote the sentence in **your** docstring that
  you think made the difference — then test it by editing that sentence and re-running.
- Did the customer get an honest "out of stock"? Paste the reply.
- Your tool and `get_product` overlap. Is that a problem? Argue it both ways in three sentences, then
  decide: would you ship both, or fold one into the other?
- `stock: 0` is a fact, not an error. What would have gone wrong if you had returned
  `{"error": "out of stock"}` instead? (Section 9 is about exactly this distinction — answer it
  from what the *model* would then be reading.)

## ⭐⭐⭐ Exercise 3 — The loop by hand, all the way

**What this is really teaching:** ownership. Section 6 walked one hop with you; `create_agent` did
four hops for you. This is you doing four hops yourself, and it is about ten lines. Participants who
finish this stop being people who *use* agents and become people who could *write* one.

Answer the multi-hop question **without `create_agent`**:

> *"Hi, I'm ayesha@example.com — what did I order most recently and has it shipped?"*

The loop you built in section 6 stopped after one round because we wrote it straight down the page.
Put a `while` around it and it is finished. The exit condition is the one you already saw: an
`AIMessage` with no `tool_calls` in it.

Three things you will need that section 6 did not:

1. **A way to find the right tool by name.** The model sends back `{"name": "list_orders", ...}` and
   you have a list of tool objects. One dict comprehension.
2. **A guard.** A loop that runs until the model stops asking is a loop that runs forever if the
   model never stops. Cap it, and say something useful when you hit the cap.
3. **Several calls per turn, possibly.** `ai.tool_calls` is a *list*. A model may ask for two things
   at once, and a loop that only runs `tool_calls[0]` silently drops the second — leaving a request
   with no matching result, which is an error the provider will report in a much less friendly way.

**✍️ Your prediction:**

*(How many times will the `while` body run? How many model calls is that in total? Write both numbers
down — the second one is the bill, and most people get it wrong by one.)*

In [ ]:
QUESTION = "Hi, I'm ayesha@example.com — what did I order most recently and has it shipped?"

model_with_tools = model.bind_tools(TOOLS)
BY_NAME = {t.name: t for t in TOOLS}

messages = [
    {"role": "system", "content": project_agent.GROUNDED_PROMPT},
    {"role": "user", "content": QUESTION},
]

# TODO 1: loop. Invoke, append the AIMessage, and break out when it has no tool_calls.

# TODO 2: inside the loop, execute EVERY tool call in ai.tool_calls — look each one up in
#         BY_NAME and pass the whole call object to .invoke() so the tool_call_id comes back
#         attached. Append each ToolMessage.

# TODO 3: cap the number of rounds (5 is plenty here) and print something honest if you hit it.

# TODO 4: print the hop number, the tool name and its arguments as you go, so you can watch it
#         work. Then print the final answer.

**✍️ Your answer.**

- How many rounds, and how many model calls? Compare against `create_agent`'s count for the same
  question in section 8.2.
- **The bill.** Print `ai.usage_metadata["input_tokens"]` on every round. The numbers go *up* each
  time. Explain why in one sentence, and say what that implies for an agent with twenty tools and a
  ten-hop task.
- What happens if you delete the system prompt from `messages`? Run it. You are removing the
  grounding instruction but keeping the tools — does it still look things up, and is the answer still
  safe?
- Now break it on purpose: execute only `ai.tool_calls[0]` on a turn where the model asked for two
  things. What does the *provider* say? (If you cannot get the model to ask for two at once, say what
  you would expect and why.)
- Finally: name one thing `create_agent` gives you that your loop does not. Read its signature —
  `create_agent(model, tools, system_prompt=..., middleware=..., response_format=..., checkpointer=...)`
  — and pick the argument you would miss first.

## ⭐⭐ Exercise 4 — Make a bad tool fail well

**What this is really teaching:** that "handle errors" is not one job. Deciding *which* failures the
model should see is a design decision, and the cost of getting it wrong is measured in what the
customer reads.

`list_orders` below is a real-looking implementation with two real-looking bugs, and it works
perfectly in testing — because in testing you always pass it a `customer_id`.

```python
@tool
def list_orders(customer_id: str) -> list[dict]:
    """Lists a customer's orders."""
    customer = CUSTOMERS[customer_id]                     # bug 1
    return [o for o in ORDERS.values() if o["customer_id"] == customer["customer_id"]]   # bug 2
```

Run it inside an agent on a question that goes through `lookup_customer` first, and find out what the
**customer** sees. Then fix it so the same failure produces a useful answer instead, and show the two
replies side by side.

Three things the fix has to decide, and the exercise is those decisions rather than the code:

1. Which of the two failures is information for the model, and which is a fault?
2. What should the error return actually *contain*? Section 9.2 has the pattern — an error that says
   what would have worked is worth more than one that only says no.
3. Is an empty list an error?

**✍️ Your prediction:**

*(What does the customer see? Pick one: a wrong answer, a polite apology, a partial answer, nothing at
all. Then say what the developer sees, which is not the same question.)*

In [ ]:
from shopwise_data import CUSTOMERS                        # noqa: E402

QUESTION = "Hi, I'm ayesha@example.com — what have I ordered?"


@tool
def list_orders_bad(customer_id: str) -> list[dict]:
    """Lists a customer's orders."""
    customer = CUSTOMERS[customer_id]
    return [o for o in ORDERS.values() if o["customer_id"] == customer["customer_id"]]


list_orders_bad.name = "list_orders"
others = [t for t in TOOLS if t.name != "list_orders"]

# TODO 1: build an agent with `others + [list_orders_bad]` and ask QUESTION.
#         Wrap it in try/except — you are going to need to.
#         Print BOTH what the customer got and what you, the developer, got.

# TODO 2: work out which of the two bugs actually fired, and why the other one did not.
#         Read the transcript: what argument did lookup_customer's result cause to be passed?

# TODO 3: write list_orders_good. Keep the same name and signature. Decide, for each failure,
#         whether the model should see it or whether your code should stop.

# TODO 4: run the same question against the fixed tool and print the two customer-facing
#         replies one after the other. That difference is the whole exercise.

**✍️ Your answer.**

- Paste both replies. What is the difference *to the customer* — not to you?
- Which bug fired, and why did the other one never get the chance? What does that tell you about how
  much of a tool's error handling is exercised by a happy-path test?
- For each of the two bugs, say whether you made it return or raise, **and why**. Use the test from
  section 9.3: can the model do anything useful with this?
- An empty list — a real customer who has genuinely never ordered anything. Error or not? Justify it,
  then check what the agent says in each case.
- Last one, and it is the one worth arguing about in class: your fixed tool returns
  `{"error": "no customer with that id"}`. A `list_orders` that is typed `-> list[dict]` now sometimes
  returns a dict. Is that acceptable? What would you do instead, and what does *that* cost?